# 82514 · Sesión S19 — SLAM: EKF-SLAM, el grafo de poses y la rejilla de ocupación

**Bloque 6** · lunes 23 de noviembre de 2026 · 2 h  ·  IQS Universitat Ramon Llull

**Qué hace este cuaderno.** El robot del almacén recorre la ruta del inventario sin mapa. Primero, el EKF de S18 con las balizas dentro del estado (EKF-SLAM): el estado crece, los errores van juntos y una asociación equivocada rompe el mapa para siempre. Después, el SLAM como mínimos cuadrados: el grafo de poses de la ruta, resuelto con el Levenberg-Marquardt de S10, con el cierre de bucle al volver al muelle y con un cierre falso. Y al final, la rejilla de ocupación en log-odds, construida con esas poses y el LiDAR de S6: el mapa del que parte S22.

**Se apoya en:** Thrun, Burgard y Fox (2005), caps. 9, 10 y 11; Lu y Milios (1997); Corke (2023), cap. 6; documentación de numpy y scipy (least_squares).

**Cómo usarlo en clase.** Sigue el guion de la sesión S19 en las diapositivas del bloque 6. Ejecuta la primera celda una sola vez al empezar; en Colab tarda unos segundos. Las celdas marcadas **Ejercicio** son para que los trabajen los estudiantes: las soluciones están al final del cuaderno.

---

In [ ]:
# Ejecutar una sola vez al abrir el cuaderno. En Colab tarda unos segundos: no instala nada.
import sys, time
import numpy as np, scipy, matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.colors import ListedColormap
from matplotlib.patches import Ellipse
from scipy.ndimage import distance_transform_edt
from scipy.optimize import least_squares

FIJAS = {'numpy': '2.4.6', 'scipy': '1.16.3', 'matplotlib': '3.10.0'}   # las de Colab con las que se probó el bloque 6
EN_COLAB = 'google.colab' in sys.modules
np.set_printoptions(precision=3, suppress=True)
plt.rcParams['figure.figsize'] = (10, 3.6)
IQS_AZUL, IQS_VERDE, GRIS, AMBAR, ROJO = '#1B2A80', '#1FA355', '#8A8FA8', '#B9770E', '#C0392B'
COMA = FuncFormatter(lambda v, p: f'{v:g}'.replace('.', ',').replace('-', '−'))   # ejes con coma decimal

def coma(x, n=2):
    """Un número con coma decimal y signo menos, para imprimir."""
    return f'{x:.{n}f}'.replace('.', ',').replace('-', '−')

def ejes(*axs):
    for ax in axs:
        ax.xaxis.set_major_formatter(COMA); ax.yaxis.set_major_formatter(COMA)

for nombre, mod in (('numpy', np), ('scipy', scipy), ('matplotlib', matplotlib)):
    print(nombre, mod.__version__, '' if mod.__version__ == FIJAS[nombre] else f'(probado con {FIJAS[nombre]})')
print('en Colab:', EN_COLAB)
if not np.__version__.startswith('2.4'):
    print('AVISO: este cuaderno se probó con numpy 2.4; las cifras con semilla pueden cambiar un poco.')

> **Aviso para Colab.** Ábrelo al acabar el taller S19a y ejecuta la primera celda: no instala nada (usa numpy, scipy y matplotlib de Colab) y tarda unos segundos.
>
> **En clase (cinco minutos):** las secciones 0 y 1 sin detenerte, y el taller, las secciones 4 y 5, que se ponen en común al acabar.
>
> **En casa:** las secciones 2, 3 y 6 (EKF-SLAM, la asociación y la rejilla) y los ejercicios 1 a 3; las soluciones están al final.
>
> **No se sube:** este cuaderno no tiene tarea en Moodle. No depende del S19a.

## 0. El almacén, el robot y el LiDAR

*En clase, ejecuta esta celda sin detenerte.* Es el simulador del cuaderno S18b (el almacén de S22, el robot con la rueda de S6, las diez balizas y su sensor de distancia y rumbo) con el LiDAR de S6 en la rejilla, y lo de hoy: EKF-SLAM, el grafo de poses y la rejilla de ocupación.

In [ ]:
# El simulador: el de S18b, más el LiDAR, EKF-SLAM, el grafo de poses y la rejilla. Sin detenerse.
import numpy as np
from scipy.ndimage import distance_transform_edt
from scipy.optimize import least_squares

D2R = np.pi / 180
RES, FILAS, COLS = 0.10, 90, 130            # celdas de 10 cm: 13 m en x, 9 m en y


def mapa_almacen():
    """La rejilla de ocupacion del almacen de S22 (True = ocupado). Fila = y/0,1; columna = x/0,1."""
    occ = np.zeros((FILAS, COLS), bool)
    occ[0, :] = occ[-1, :] = True              # paredes exteriores
    occ[:, 0] = occ[:, -1] = True
    occ[42:47, :] = True                       # el muro central, de y = 4,2 a 4,7 m
    occ[42:47, 8:28] = False                   #   la puerta ancha, de x = 0,8 a 2,8 m
    occ[42:47, 92:100] = False                 #   la puerta estrecha, de x = 9,2 a 10,0 m
    occ[12:36, 40:47] = True                   # estanterias de la nave de abajo
    occ[12:36, 64:71] = True
    occ[58:78, 34:41] = True                   # estanterias de la nave de arriba
    occ[58:78, 58:65] = True
    occ[58:78, 82:89] = True
    return occ


OCC = mapa_almacen()
MUELLE, PICKING = np.array([0.6, 0.6]), np.array([8.3, 8.2])
BALIZAS = np.array([[0.1, 2.0], [3.0, 0.1], [8.0, 0.1], [12.9, 2.0], [5.5, 4.15],      # nave de abajo
                    [0.1, 7.0], [4.7, 8.9], [10.5, 8.9], [12.9, 6.5], [7.0, 4.75]])    # nave de arriba
RUTA = np.array([[0.6, 0.6], [1.8, 0.6], [1.8, 5.2], [9.6, 5.2], [9.6, 0.6], [0.6, 0.6]])   # la ruta del inventario

V, HZ = 0.5, 10                              # 0,5 m/s; odometria a 10 Hz
DD = V / HZ                                  # 5 cm por paso
SIG_D, SIG_T = 0.06 / np.sqrt(20), 0.2 * D2R # por paso: 13,4 mm y 0,2° (6 cm y 0,9° por metro)
Q = np.diag([SIG_D**2, SIG_T**2])            # Q: la covarianza del ruido de proceso (la odometria)
SIG_R, SIG_PHI, ALCANCE = 0.05, 1.0 * D2R, 8.0
R = np.diag([SIG_R**2, SIG_PHI**2])          # R: la covarianza del ruido de medida (el sensor de balizas)
CADA = 10                                    # un barrido de balizas cada 10 pasos: una vez por segundo
K95 = 2.4477                                 # en el plano, la elipse de 2,45σ encierra el 95 %


def envuelve(a):
    """Un angulo en (−π, π]: sin esto, la innovacion del rumbo salta 360°."""
    return (a + np.pi) % (2 * np.pi) - np.pi


def paso(x, d):
    """El modelo de movimiento: la pose x = (x; y; θ) compuesta con la odometria d = (δ_d; δ_θ) en SE(2),
    como en S8: avanzar δ_d en la direccion del rumbo y despues girar δ_θ."""
    return np.array([x[0] + d[0] * np.cos(x[2]), x[1] + d[0] * np.sin(x[2]), envuelve(x[2] + d[1])])


def jac_paso(x, d):
    """Sus jacobianos: F_x respecto a la pose (3 x 3) y F_v respecto a la odometria (3 x 2)."""
    c, s = np.cos(x[2]), np.sin(x[2])
    Fx = np.array([[1.0, 0.0, -d[0] * s], [0.0, 1.0, d[0] * c], [0.0, 0.0, 1.0]])
    Fv = np.array([[c, 0.0], [s, 0.0], [0.0, 1.0]])
    return Fx, Fv


def h(x, b):
    """El modelo de medida: distancia y rumbo a la baliza b desde la pose x."""
    dx, dy = b[0] - x[0], b[1] - x[1]
    return np.array([np.hypot(dx, dy), envuelve(np.arctan2(dy, dx) - x[2])])


def jac_h(x, b):
    """Su jacobiano H_x respecto a la pose (2 x 3)."""
    dx, dy = b[0] - x[0], b[1] - x[1]
    r2 = dx * dx + dy * dy
    r = np.sqrt(r2)
    return np.array([[-dx / r, -dy / r, 0.0], [dy / r2, -dx / r2, -1.0]])


def libre_vista(a, b):
    """¿Ve el sensor la baliza b desde a? Sin celdas ocupadas en medio (la baliza esta en la pared)."""
    n = int(np.ceil(np.linalg.norm(b - a) / 0.02))
    p = a + np.linspace(0, 1, n)[:, None] * (b - a)
    cc = np.clip((p[:, 0] / RES).astype(int), 0, COLS - 1)
    ff = np.clip((p[:, 1] / RES).astype(int), 0, FILAS - 1)
    return not OCC[ff, cc][:-6].any()


def ruta_densa():
    """La ruta del inventario, en pasos de 5 cm."""
    pts = [RUTA[0]]
    for a, b in zip(RUTA[:-1], RUTA[1:]):
        n = int(round(np.linalg.norm(b - a) / DD))
        pts += [a + (b - a) * k / n for k in range(1, n + 1)]
    return np.array(pts)


def mandos(rd):
    """La odometria verdadera de cada paso: el avance y el giro que llevan de un punto de la ruta al siguiente."""
    d, th = [], 0.0
    for a, b in zip(rd[:-1], rd[1:]):
        thn = np.arctan2(b[1] - a[1], b[0] - a[0])
        d.append((np.linalg.norm(b - a), envuelve(thn - th)))
        th = thn
    return np.array(d)


RD = ruta_densa()
U = mandos(RD)                               # 544 pasos, 27,2 m
VERDAD = [np.array([RD[0, 0], RD[0, 1], 0.0])]
for d in U:
    VERDAD.append(paso(VERDAD[-1], d))
VERDAD = np.array(VERDAD)
VISIBLES = [[j for j, b in enumerate(BALIZAS) if np.linalg.norm(b - p[:2]) < ALCANCE and libre_vista(p[:2], b)]
            for p in VERDAD]


def simula(semilla=1):
    """Lo que leen los sensores en una pasada por la ruta: la odometria de cada paso y, una vez por segundo,
    la distancia y el rumbo de cada baliza visible (con su numero: la asociacion se supone conocida)."""
    rng = np.random.default_rng(semilla)
    odo = U + rng.normal(0, [SIG_D, SIG_T], size=U.shape)
    med = {}
    for k in range(CADA, len(VERDAD), CADA):
        med[k] = [(j, h(VERDAD[k], BALIZAS[j]) + rng.normal(0, [SIG_R, SIG_PHI])) for j in VISIBLES[k]]
    return odo, med
# =====================================================================================================
# S19 · 2. El LiDAR de S6 en la rejilla, el campo de verosimilitud y MCL en el almacen
# =====================================================================================================
ANG = np.radians(np.arange(0, 360, 0.25))    # 1440 haces: 360° con 0,25° de resolucion
RMAX, SIG_L, P_SIN = 10.0, 0.02, 0.02        # alcance de 10 m, 2 cm de ruido, 2 % de rayos sin retorno
HACES = np.arange(0, ANG.size, 80)           # el filtro usa 18 haces, uno cada 20°
DIST = distance_transform_edt(~OCC) * RES    # el campo de verosimilitud: distancia de cada celda al obstaculo
LIBRE = DIST > 0.28                          # donde cabe el centro del robot (radio de 0,28 m, el de S22)
_FL, _CL = np.nonzero(LIBRE)


def rayos(pose, angs, occ=OCC):
    """La distancia al primer obstaculo de cada rayo, marchando a pasos de 1 cm por la rejilla."""
    t = np.arange(0.01, RMAX, 0.01)
    a = pose[2] + angs
    cc = ((pose[0] + np.outer(np.cos(a), t)) / RES).astype(int)
    ff = ((pose[1] + np.outer(np.sin(a), t)) / RES).astype(int)
    fuera = (cc < 0) | (cc >= COLS) | (ff < 0) | (ff >= FILAS)
    choca = fuera | occ[np.clip(ff, 0, FILAS - 1), np.clip(cc, 0, COLS - 1)]
    r = t[choca.argmax(axis=1)]
    r[~choca.any(axis=1)] = RMAX
    return r


def barrido(pose, rng, idx=None):
    """Un barrido del LiDAR desde la pose (los haces idx; todos si no se dan), con su ruido y sus rayos perdidos."""
    a = ANG if idx is None else ANG[idx]
    r = rayos(pose, a) + rng.normal(0, SIG_L, a.size)
    r[rng.random(a.size) < P_SIN] = RMAX
    return np.minimum(r, RMAX)


def relativa(a, b):
    """La pose de b vista desde a, en SE(2) (la ⊖ de S8): adelante, a la izquierda y el giro."""
    c, s = np.cos(a[2]), np.sin(a[2])
    d = np.asarray(b[:2]) - np.asarray(a[:2])
    return np.array([c * d[0] + s * d[1], -s * d[0] + c * d[1], envuelve(b[2] - a[2])])
# =====================================================================================================
# S19 · 4. EKF-SLAM: la pose y las balizas en un solo estado, de 3 + 2N componentes
# =====================================================================================================
PUERTA_99 = 9.21                             # χ² con 2 grados de libertad: el 99 % de las medidas buenas, por debajo
SIGMA0 = np.diag([0.02**2, 0.02**2, (1.0 * D2R)**2])   # al salir del muelle: 2 cm y 1° (como en S18b)


def ekf_slam(odo, med, cambia=None, asociar=None, puerta=PUERTA_99, x0=None):
    """EKF-SLAM con la ruta del inventario. El estado es (x; y; θ; m1x; m1y; …): cada baliza nueva lo alarga en
    dos. med[k] = [(j, z)]: j es el numero de la baliza (la asociacion, conocida) salvo que asociar=True, y entonces
    se busca la del mapa con menor d² y, si ninguna pasa la puerta, se toma por nueva.
    cambia = (k, j_real, j_usada): en el barrido k, la medida de j_real se atribuye a j_usada (un error).
    Devuelve el estado y la covarianza en cada paso, el orden en que se anadieron las balizas y las d²."""
    x = (VERDAD[0] if x0 is None else np.asarray(x0)).astype(float).copy()
    S_ = SIGMA0.copy()
    orden, hist, D2 = [], [], []
    for k, d in enumerate(odo, start=1):
        n = x.size
        Fx, Fv = jac_paso(x[:3], d)                           # prediccion: solo se mueve la pose
        x[:3] = paso(x[:3], d)
        F = np.eye(n); F[:3, :3] = Fx
        S_ = F @ S_ @ F.T
        S_[:3, :3] += Fv @ Q @ Fv.T
        for j, z in med.get(k, []):
            if cambia and k == cambia[0] and j == cambia[1]:
                j = cambia[2]
            if asociar:
                j = None
                mejor = (np.inf, None)
                for i, ji in enumerate(orden):
                    nu, Hf, Sm = _innov(x, S_, i, z)
                    d2 = float(nu @ np.linalg.solve(Sm, nu))
                    if d2 < mejor[0]:
                        mejor = (d2, ji)
                if mejor[0] < puerta:
                    j = mejor[1]
                else:
                    j = ('nueva', len(orden))
            if j not in orden:                                # baliza nueva: se anade al estado
                th = x[2] + z[1]
                m = x[:2] + z[0] * np.array([np.cos(th), np.sin(th)])
                Gx = np.array([[1, 0, -z[0] * np.sin(th)], [0, 1, z[0] * np.cos(th)]])
                Gz = np.array([[np.cos(th), -z[0] * np.sin(th)], [np.sin(th), z[0] * np.cos(th)]])
                Sxx = S_[:3, :3]
                Smm = Gx @ Sxx @ Gx.T + Gz @ R @ Gz.T
                Smx = Gx @ S_[:3, :]
                S_ = np.block([[S_, Smx.T], [Smx, Smm]])
                x = np.concatenate([x, m])
                orden.append(j)
                continue
            i = orden.index(j)
            nu, Hf, Sm = _innov(x, S_, i, z)
            D2.append((k, j, float(nu @ np.linalg.solve(Sm, nu))))
            L = S_ @ Hf.T @ np.linalg.inv(Sm)                 # la ganancia, ahora de (3 + 2N) x 2
            x = x + L @ nu
            x[2] = envuelve(x[2])
            S_ = (np.eye(x.size) - L @ Hf) @ S_
        hist.append((x.copy(), S_.copy()))
    return hist, orden, D2


def _innov(x, S_, i, z):
    """La innovacion de la baliza i del estado, su jacobiano respecto a todo el estado y su covarianza."""
    b = x[3 + 2 * i: 5 + 2 * i]
    Hx = jac_h(x[:3], b)
    Hf = np.zeros((2, x.size))
    Hf[:, :3] = Hx
    Hf[:, 3 + 2 * i: 5 + 2 * i] = -Hx[:, :2]
    nu = z - h(x[:3], b)
    nu[1] = envuelve(nu[1])
    return nu, Hf, Hf @ S_ @ Hf.T + R


def error_mapa(x, orden):
    """Error medio y maximo de las balizas del mapa frente a las de verdad (solo las de numero conocido)."""
    n = (x.size - 3) // 2
    e = [np.hypot(*(x[3 + 2 * i: 5 + 2 * i] - BALIZAS[j])) for i, j in enumerate(orden[:n])
         if isinstance(j, (int, np.integer))]
    return (float(np.mean(e)), float(np.max(e))) if e else (np.nan, np.nan)
# =====================================================================================================
# S19 · 5. El grafo de poses: un nodo por metro, una arista por tramo de odometria y el cierre de bucle
# =====================================================================================================
def odometria_integrada(odo, x0=None):
    """La pose que da la odometria sola, paso a paso (a estima)."""
    X = [VERDAD[0].copy() if x0 is None else np.asarray(x0, float)]
    for d in odo:
        X.append(paso(X[-1], d))
    return np.array(X)


CADA_NODO = 20                               # un nodo cada 20 pasos: cada metro de ruta
# la arista de odometria, isotropa por sencillez: de lado, la odometria da 0,9 cm por metro
S_ARISTA = np.array([SIG_D * np.sqrt(CADA_NODO), SIG_D * np.sqrt(CADA_NODO), SIG_T * np.sqrt(CADA_NODO)])
S_CIERRE = np.array([0.02, 0.02, 0.5 * D2R]) # el cierre: el barrido de ahora alineado con el del principio


def nodos_de(X):
    """Los indices de los nodos (cada metro y el ultimo paso) y sus poses."""
    idx = list(range(0, len(X), CADA_NODO))
    if idx[-1] != len(X) - 1:
        idx.append(len(X) - 1)
    return idx, X[idx]


def pasada_grafo(semilla=20):
    """Una pasada por la ruta para el grafo: la odometria de cada paso y la medida del cierre al volver al
    muelle (la pose del ultimo nodo vista desde el primero, con 2 cm y 0,5° de error)."""
    rng = np.random.default_rng(semilla)
    odo = U + rng.normal(0, [SIG_D, SIG_T], size=U.shape)
    idx, Nv = nodos_de(VERDAD)
    cierre = relativa(Nv[0], Nv[-1]) + rng.normal(0, S_CIERRE)
    return odo, (0, len(idx) - 1, cierre)


def grafo(X_odo, cierres=()):
    """Arma el grafo con las poses de la odometria: aristas de odometria entre nodos seguidos y las de cierre
    (i, j, medida). Resuelve min Σ eᵀ·Ω·e con el LM de S10 (least_squares, method='lm'). El primer nodo se fija."""
    idx, N = nodos_de(X_odo)
    aristas = [(i, i + 1, relativa(N[i], N[i + 1]), S_ARISTA) for i in range(len(idx) - 1)]
    aristas += [(i, j, m, S_CIERRE) for i, j, m in cierres]

    def residuos(p):
        Xn = np.vstack([N[0], p.reshape(-1, 3)])
        r = []
        for i, j, m, s in aristas:
            e = relativa(Xn[i], Xn[j]) - m
            e[2] = envuelve(e[2])
            r.append(e / s)
        return np.concatenate(r)

    sol = least_squares(residuos, N[1:].ravel(), method="lm")
    Xg = np.vstack([N[0], sol.x.reshape(-1, 3)])
    r = residuos(sol.x).reshape(-1, 3)
    return dict(idx=idx, N_odo=N, N=Xg, aristas=aristas, chi2=(r**2).sum(axis=1), sol=sol)
# =====================================================================================================
# S19 · 6. La rejilla de ocupacion en log-odds, con las poses y el LiDAR
# =====================================================================================================
L_OCC = np.log(0.7 / 0.3)                    # un impacto: +0,847
L_LIB = np.log(0.3 / 0.7)                    # el rayo pasa por la celda: −0,847
L_TOPE = 5.0                                 # los log-odds se limitan a ±5 (p de 0,007 a 0,993)


def rejilla_ocupacion(poses, barridos, idx=None):
    """Cada celda, una puerta de Bayes binaria en log-odds: el rayo la deja libre hasta el impacto (−0,847 por
    rayo) y la ocupa en el impacto (+0,847). Los rayos sin retorno solo dejan libre."""
    l = np.zeros((FILAS, COLS))
    a0 = ANG if idx is None else ANG[idx]
    for pose, r in zip(poses, barridos):
        a = pose[2] + a0
        for ang, rr in zip(a, r):
            t = np.arange(0.0, rr, 0.05)
            cc = ((pose[0] + t * np.cos(ang)) / RES).astype(int)
            ff = ((pose[1] + t * np.sin(ang)) / RES).astype(int)
            ok = (cc >= 0) & (cc < COLS) & (ff >= 0) & (ff < FILAS)
            cel = np.unique(ff[ok] * COLS + cc[ok])
            if rr < RMAX - 1e-6:
                ci = int((pose[0] + rr * np.cos(ang)) / RES); fi = int((pose[1] + rr * np.sin(ang)) / RES)
                if 0 <= ci < COLS and 0 <= fi < FILAS:
                    imp = fi * COLS + ci
                    cel = cel[cel != imp]
                    l.flat[imp] += L_OCC
            l.flat[cel] += L_LIB
        np.clip(l, -L_TOPE, L_TOPE, out=l)
    return np.round(l, 9)        # un impacto y un paso se anulan: 0 exacto (sin el redondeo, ±10⁻¹⁶ decidiria la celda)


def compara_rejilla(l):
    """La rejilla construida frente al mapa de S22, en las celdas vistas: el % bien clasificadas, y las celdas
    ocupadas fantasma (marcadas ocupadas a mas de 10 cm de cualquier obstaculo verdadero)."""
    vista = l != 0               # p distinta de 0,5: alguna evidencia neta
    ocup = l > 0
    bien = (ocup == OCC) & vista
    fantasma = ocup & (DIST > 0.15)
    return dict(vistas=int(vista.sum()), bien=100 * bien.sum() / vista.sum(),
                ocupadas=int(ocup.sum()), fantasma=int(fantasma.sum()),
                pct_fantasma=100 * fantasma.sum() / max(ocup.sum(), 1))

MAPA = ListedColormap(['white', '#4A4F63'])

def dibuja_almacen(ax, alfa=1.0, ocupado=None):
    ax.imshow(OCC if ocupado is None else ocupado, cmap=MAPA, origin='lower', extent=(0, 13, 0, 9), interpolation='nearest', alpha=alfa)
    ax.set_xlim(0, 13); ax.set_ylim(0, 9); ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])

print(f'Almacén de {COLS*RES:.0f} x {FILAS*RES:.0f} m ; {len(BALIZAS)} balizas ; ruta de {len(U)} pasos ({coma(len(U)*DD, 1)} m)')

## 1. Una pasada por el almacén, sin mapa

*En clase, sin detenerte.* Es lo de la diapositiva 76. El robot recorre la ruta del inventario (semilla 20): la odometría de cada paso y, una vez por segundo, la distancia y el rumbo de cada baliza visible. Con la odometría sola, cada baliza cae donde la pone la pose de ese momento: la misma baliza, en sitios distintos. Para hacer el mapa hace falta la pose, y para saber la pose hace falta el mapa: es el problema circular del SLAM.

In [ ]:
odo, med = simula(20)
Xo = odometria_integrada(odo)
e = np.hypot(*(Xo[:, :2] - VERDAD[:, :2]).T)
print(f'la odometría sola: error final {coma(e[-1]*100, 0)} cm, máximo {coma(e.max()*100, 0)} cm')
fig, ax = plt.subplots(figsize=(7, 5))
dibuja_almacen(ax, alfa=0.35)
ax.plot(VERDAD[:, 0], VERDAD[:, 1], color=ROJO, label='la ruta de verdad')
ax.plot(Xo[:, 0], Xo[:, 1], '--', color=IQS_AZUL, label='la odometría')
for k in sorted(med):
    for j, z in med[k]:
        th = Xo[k][2] + z[1]
        ax.plot(*(Xo[k][:2] + z[0]*np.array([np.cos(th), np.sin(th)])), '.', color=AMBAR, ms=3)
ax.plot(BALIZAS[:, 0], BALIZAS[:, 1], 'P', color=IQS_VERDE, ms=9, label='las balizas')
ax.legend(fontsize=8, loc='upper right'); plt.show()

## 2. EKF-SLAM: el estado crece con cada baliza

*En casa.* Es lo de la diapositiva 77. El estado es la pose y cada baliza, (x; y; θ; m₁ₓ; m₁ᵧ; …): 3 + 2N componentes. Al predecir solo se mueve la pose. Cuando aparece una baliza nueva, se añade al estado en la posición que dice la medida, m = (x; y) + r·(cos(θ + φ); sen(θ + φ)), con su covarianza y sus correlaciones con todo lo demás (los jacobianos G_x y G_z de esa fórmula). Al volver a ver una baliza, se corrige con el EKF de S18, pero H ahora tiene 3 + 2N columnas: la medida depende de la pose y de esa baliza, y la ganancia reparte la corrección por todo el estado.

In [ ]:
t0 = time.time()
hist, orden, D2 = ekf_slam(odo, med)
x, Sigma = hist[-1]
em = error_mapa(x, orden)
ep = np.array([np.hypot(*(h_[0][:2] - VERDAD[k + 1][:2])) for k, h_ in enumerate(hist)])
print(f'{coma(time.time() - t0, 2)} s ; estado final de {x.size} componentes, Σ de {x.size}x{x.size}')
print(f'mapa: error medio {coma(em[0]*100, 1)} cm, máximo {coma(em[1]*100, 1)} cm ; pose: RMSE {coma(np.sqrt((ep**2).mean())*100, 1)} cm, final {coma(ep[-1]*100, 1)} cm')
for k in (269, 270):
    xx = hist[k - 1][0]
    print(f'paso {k}: mapa, error medio {coma(error_mapa(xx, orden)[0]*100, 1)} cm y máximo {coma(error_mapa(xx, orden)[1]*100, 1)} cm')
# el mismo barrido del paso 270 sin la medida de la (8,0; 0,1): ¿quién corrige a las dos de arriba a la derecha?
med_sin = dict(med); med_sin[270] = [(j, z) for j, z in med[270] if j != 2]
hist_sin, _, _ = ekf_slam(odo, med_sin)
for j in (8, 7):
    i = orden.index(j)
    e_ = [np.hypot(*(hh[3+2*i:5+2*i] - BALIZAS[j])) * 100 for hh in (hist[268][0], hist[269][0], hist_sin[269][0])]
    print(f'la ({coma(BALIZAS[j][0], 1)}; {coma(BALIZAS[j][1], 1)}): {coma(e_[0], 1)} cm en el 269, {coma(e_[1], 1)} en el 270 ; sin la medida de la (8,0; 0,1), {coma(e_[2], 1)}')
rho = Sigma / np.sqrt(np.outer(np.diag(Sigma), np.diag(Sigma)))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={'width_ratios': [1.4, 1]})
dibuja_almacen(a1, alfa=0.35)
a1.plot([h_[0][0] for h_ in hist], [h_[0][1] for h_ in hist], color=IQS_AZUL, lw=1)
for i in range(len(orden)):
    w, v = np.linalg.eigh(Sigma[3+2*i:5+2*i, 3+2*i:5+2*i])
    a1.add_patch(Ellipse(x[3+2*i:5+2*i], 2*2.45*np.sqrt(w[1]), 2*2.45*np.sqrt(w[0]), angle=np.degrees(np.arctan2(v[1, 1], v[0, 1])), fc=AMBAR, alpha=0.4))
a1.plot(BALIZAS[:, 0], BALIZAS[:, 1], 'P', color=IQS_VERDE, ms=8)
a2.imshow(np.abs(rho), cmap='Blues', vmin=0, vmax=1); a2.set_title('|correlación| en Σ', fontsize=10)
plt.tight_layout(); plt.show()
for N in (10, 100, 1000):
    n = 3 + 2*N
    print(f'con {N} balizas: estado de {n}, Σ de {n*n} entradas, {coma(n*n*8/1e6, 2)} MB')

El mapa acaba con 9,7 cm de error medio y la pose con 6,3 cm, frente a los 52 de la odometría sola. Al volver a ver en el paso 270 la baliza (8,0; 0,1), que no veía desde el paso 40, las dos de arriba a la derecha, (12,9; 6,5) y (10,5; 8,9), bajan de 32 y 26 cm a 18 en un barrido; con el mismo barrido sin esa medida, apenas se mueven. Los errores de las balizas van juntos (la matriz de correlaciones está casi llena) y se corrigen juntos. El precio: Σ crece con el cuadrado del número de balizas, y cada corrección la toca entera.

## 3. ¿Es la baliza de antes? La asociación de datos

*En casa.* Es lo de la diapositiva 78. Un reflector no dice cuál es. Primero, **una sola medida atribuida a la baliza equivocada** (en el barrido del paso 310, la (8,0; 0,1) tomada por la (0,1; 2,0)). Después, la asociación de verdad: cada medida va a la baliza del mapa con menor d² = νᵀ·S⁻¹·ν, y si ninguna pasa la puerta de S18 (9,21), es una baliza nueva.

In [ ]:
k_mal = [k for k in sorted(med) if len(med[k]) >= 2][30]
hb, ob, Db = ekf_slam(odo, med, cambia=(k_mal, med[k_mal][0][0], 0))
print(f'una medida mal asociada en el paso {k_mal}: su d² = {coma([d for d in Db if d[0] == k_mal][0][2], 0)} ;'
      f' el mapa acaba con {coma(error_mapa(hb[-1][0], ob)[0]*100, 0)} cm de error medio (sin el error, {coma(em[0]*100, 1)})')
d2 = np.array([d[2] for d in D2])
print(f'las {d2.size} medidas bien asociadas: d² medio {coma(d2.mean())} ; por encima de 9,21: {int((d2 > 9.21).sum())} (se espera el 1 %, {coma(0.01*d2.size, 1)}):',
      ', '.join(f'paso {k}, d² = {coma(d, 1)}' for k, j, d in D2 if d > 9.21))
ha, oa, _ = ekf_slam(odo, med, asociar=True)
xa = ha[-1][0]
n = (xa.size - 3)//2
print(f'sin saber cuál es: el mapa acaba con {n} balizas')
for i in range(n):
    m = xa[3+2*i:5+2*i]
    j = int(np.argmin(np.hypot(*(BALIZAS - m).T)))
    print(f'  ({coma(m[0])}; {coma(m[1])}) -> la de verdad más cercana, ({coma(BALIZAS[j][0], 1)}; {coma(BALIZAS[j][1], 2)})')

Una sola asociación equivocada, sin puerta, deja el mapa con más de 2 m de error medio: en S18, con el mapa dado, se corregía en dos segundos; en el SLAM, el error se queda en el mapa. Su d² era de miles: la puerta la habría descartado. Pero la puerta deja fuera el 1 % de las medidas buenas (es lo que significa el 99 %): en el paso 30, una medida de la (8,0; 0,1) da d² = 13,8, no pasa la puerta y se toma por nueva. El mapa acaba con once balizas. Remedios: dos umbrales (aceptar por debajo de 5,99 y crear por encima de 13,8) o esperar varias medidas antes de crear una baliza.

## 4. Taller: el grafo de poses, sin cierre y con él

*En clase.* Es lo de la diapositiva 79. Un nodo por metro de ruta (29), con la pose que da la odometría; una arista de odometría entre cada dos nodos seguidos (28), con su covarianza; y la del cierre: al volver al muelle, alinear el barrido de ahora con el del principio da la pose del último nodo vista desde el primero, con 2 cm y 0,5°. El residuo de una arista es la pose del nodo j vista desde el i (como en S8) menos lo medido, y se minimiza la suma de los residuos al cuadrado, cada uno pesado con la inversa de su covarianza, con el Levenberg-Marquardt de S10 (`least_squares(method='lm')`), como la calibración de S16.

**¿Cuánto error tiene cada nodo sin el cierre y con él? ¿Dónde va el error al cerrar?**

In [ ]:
odoG, cierre = pasada_grafo(20)
XoG = odometria_integrada(odoG)
idx, Nv = nodos_de(VERDAD)
t0 = time.time()
sin = grafo(XoG, ())
con = grafo(XoG, [cierre])
print(f'{len(idx)} nodos, {len(con["aristas"])} aristas, {3*(len(idx) - 1)} incógnitas ; {coma((time.time() - t0)/2, 2)} s cada una')
e_odo = np.hypot(*(con['N_odo'][:, :2] - Nv[:, :2]).T)
e_sin = np.hypot(*(sin['N'][:, :2] - Nv[:, :2]).T)
e_con = np.hypot(*(con['N'][:, :2] - Nv[:, :2]).T)
for nom, e in (('solo odometría', e_odo), ('grafo sin cierre', e_sin), ('grafo con el cierre', e_con)):
    print(f'{nom:20s}: final {coma(e[-1]*100, 1)} cm, máximo {coma(e.max()*100, 1)} cm, medio {coma(e.mean()*100, 1)} cm')
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
dibuja_almacen(a1, alfa=0.3)
a1.plot(Nv[:, 0], Nv[:, 1], color=ROJO, label='verdad'); a1.plot(*con['N_odo'][:, :2].T, '--o', ms=3, color=GRIS, label='odometría')
a1.plot(*con['N'][:, :2].T, '-o', ms=3, color=IQS_AZUL, label='grafo con el cierre'); a1.legend(fontsize=8)
a2.plot(e_odo*100, '-o', ms=3, color=GRIS, label='odometría'); a2.plot(e_con*100, '-o', ms=3, color=IQS_AZUL, label='con el cierre')
a2.set_xlabel('nodo (cada metro)'); a2.set_ylabel('error [cm]'); a2.legend(); ejes(a2)
plt.tight_layout(); plt.show()

Sin el cierre, el grafo no tiene nada que corregir: da lo mismo que la odometría (52 cm al final, 94 cm de máximo). Con el cierre, el último nodo queda a 2,9 cm y el peor a 27 cm: **el error no desaparece, se reparte por toda la vuelta**, cada arista cede un poco, en proporción a su incertidumbre.

## 5. Taller: un cierre falso

*En clase.* Es lo de la diapositiva 81. Dos tramos de pasillo se parecen: el de arriba hacia el este (nodo 10, el pasillo junto al muro) y el de abajo hacia el oeste (nodo 21) dan barridos casi iguales girados 180°, y la alineación dice «el mismo sitio, girado 180°». **¿Qué le pasa al mapa? ¿Y a los residuos, eᵀ·Ω·e?**

Para juzgar la suma de los residuos hace falta saber cuántos **sobran**: cada arista da 3 residuos y el grafo tiene 84 incógnitas. Si el modelo es bueno, la suma sigue una χ² con (residuos − incógnitas) grados de libertad.

In [ ]:
from scipy.stats import chi2
ia, ie = 10, 21                                   # el pasillo de arriba (este) y el de abajo (oeste)
m_falsa = np.array([0.0, 0.0, np.pi])
falso = grafo(XoG, [cierre, (ia, ie, m_falsa)])
sin_bueno = grafo(XoG, [(ia, ie, m_falsa)])
e_f = np.hypot(*(falso['N'][:, :2] - Nv[:, :2]).T)
print(f'nodos {ia} y {ie} ; con el cierre falso: error máximo {coma(e_f.max())} m, final {coma(e_f[-1]*100, 1)} cm')
sobran = lambda g: 3 * len(g['aristas']) - 3 * (len(g['idx']) - 1)
for nom, g in (('con el cierre bueno', con), ('con el bueno y el falso', falso)):
    print(f"{nom}: suma {coma(g['chi2'].sum(), 1)} ; sobran {sobran(g)} residuos ; el 99 % queda por debajo de {coma(chi2.ppf(0.99, sobran(g)), 1)}")
print(f"la arista falsa sola: {coma(falso['chi2'][-1], 1)} ; las de odometría: {coma(falso['chi2'][:-2].mean(), 1)} de media")
print(f"quitando el falso: {coma(con['chi2'].sum(), 1)} ; quitando el bueno: {coma(sin_bueno['chi2'].sum(), 0)}")
fig, ax = plt.subplots(figsize=(7, 5))
dibuja_almacen(ax, alfa=0.3)
ax.plot(Nv[:, 0], Nv[:, 1], color=ROJO); ax.plot(*falso['N'][:, :2].T, '-o', ms=3, color=IQS_AZUL)
ax.set_xlim(min(-0.3, falso['N'][:, 0].min() - 0.5), 13.3); ax.set_ylim(min(-0.3, falso['N'][:, 1].min() - 0.5), 9.3); plt.show()

El final sigue bien (5 cm), pero el mapa se dobla: hasta 2,7 m de error. La arista falsa sola da 10,9, menos que las de odometría, porque el LM reparte su error por todo el grafo: su residuo no la delata. La suma, sí: 761, cuando con 6 residuos de sobra debería quedar por debajo de 16,8. Y quitando cada cierre se ve cuál sobra. **Antes de aceptar un cierre, se mira cuánto sube la suma.**

## 6. La rejilla de ocupación: el mapa de S22

*En casa.* Es lo de la diapositiva 82. Con las poses de cada nodo y un barrido del LiDAR en cada uno, cada celda es una **puerta de Bayes**, ocupada o libre, como la de S18, que se lleva en **log-odds**, l = ln(p/(1 − p)): cada rayo deja libres las celdas que cruza (−0,847, la de p = 0,3) y ocupa la del final (+0,847, la de p = 0,7). Tres impactos dan l = 2,54, p = 0,927. Se compara la rejilla que sale con las poses de la odometría y con las del grafo.

In [ ]:
rng = np.random.default_rng(55)
barridos = [barrido(p, rng) for p in Nv]                 # lo que ve el LiDAR desde cada nodo de verdad
sub = np.arange(0, ANG.size, 2)                         # uno de cada dos haces: 720
fig, axs = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, (nom, poses) in zip(axs, (('poses de la odometría', con['N_odo']), ('poses del grafo', con['N']), ('poses verdaderas', Nv))):
    l = rejilla_ocupacion(poses, [b[sub] for b in barridos], idx=sub)
    cmp_ = compara_rejilla(l)
    ax.imshow(1/(1 + np.exp(l)), cmap='gray', origin='lower', extent=(0, 13, 0, 9), vmin=0, vmax=1)
    ax.set_title(f"{nom}: {coma(cmp_['pct_fantasma'], 0)} % de paredes fantasma", fontsize=9); ax.set_xticks([]); ax.set_yticks([])
    print(f"{nom:22s}: bien clasificadas {coma(cmp_['bien'], 1)} % ; ocupadas {cmp_['ocupadas']}, a más de 10 cm de una pared de verdad {cmp_['fantasma']}")
plt.tight_layout(); plt.show()

Con las poses de la odometría, dos de cada tres celdas que la rejilla marca ocupadas no son pared: las paredes salen dobles y torcidas. Con las del grafo, una de cada nueve. Es el mapa del que parte S22, el jueves 3 de diciembre, y que Nav2 convierte en mapa de coste.

### Ejercicio 1 (en casa): balizas de corto alcance

En la sección 2, quédate solo con las medidas de menos de 4 m (`med4 = {k: [(j, z) for j, z in v if z[0] < 4] for k, v in med.items()}`) y repite el EKF-SLAM. ¿Cuántas medidas quedan? ¿Qué les pasa al mapa y a la pose?

In [ ]:
# Ejercicio 1
# med4 = {k: [(j, z) for j, z in v if z[0] < 4] for k, v in med.items()}
# hist4, orden4, _ = ekf_slam(odo, med4)

### Ejercicio 2 (en casa): un cierre que se fía poco

En la sección 4, multiplica `S_CIERRE` por 10 y por 100 (el cierre medido con 20 cm y 5°, y con 2 m y 50°) y repite el grafo con el cierre. ¿Cuánto corrige ahora? Deja después `S_CIERRE` como estaba.

In [ ]:
# Ejercicio 2
# S_CIERRE = np.array([0.02, 0.02, 0.5*D2R]) * 10
# ... grafo(XoG, [cierre]) ...
# S_CIERRE = np.array([0.02, 0.02, 0.5*D2R])

### Ejercicio 3 (en casa): una rejilla más prudente

En la sección 6, cambia la verosimilitud de cada impacto de 0,7 a 0,6 (`L_OCC = np.log(0.6/0.4)` y `L_LIB = -L_OCC`) y repite las rejillas. ¿Cambian las paredes fantasma? ¿Qué arregla las poses y qué arregla el modelo del sensor?

In [ ]:
# Ejercicio 3
# L_OCC = np.log(0.6/0.4); L_LIB = -L_OCC
# ... repite la celda de la sección 6 ...
# L_OCC = np.log(0.7/0.3); L_LIB = np.log(0.3/0.7)

---

## Soluciones

**Ejercicio 1.** Quedan 103 medidas de 182 (ningún barrido se queda sin ninguna) y se ven las diez balizas, pero con menos medidas y peor repartidas: el mapa pasa de 9,7 a 18,6 cm de error medio (de 17,8 a 32,0 de máximo) y la pose, de 13,0 a 33,9 cm de RMSE (máximo 57,8). En EKF-SLAM, cada baliza que se deja de ver deja de corregir; lo que no se mide, se adivina con la odometría.

**Ejercicio 2.** Con σ diez veces mayor (20 cm y 5°), el último nodo queda a 3,7 cm, pero el error máximo sube de 27 a 64,5 cm y el medio de 12 a 37 cm: el cierre pesa menos frente a la odometría y reparte menos. Con σ cien veces mayor, casi no corrige: 47,9 cm al final y 93 de máximo, como la odometría sola. Lo que se fía de una arista lo decide su covarianza: un cierre mal medido no sirve.

**Ejercicio 3.** Con p = 0,6 las paredes fantasma apenas cambian: con la odometría, el 62 % (323 de 517) frente al 66 % (347 de 523); con el grafo, el 12 % (71 de 616) frente al 11 % (66 de 623). Las celdas bien clasificadas: 83,6 % y 87,8 %, frente a 83,3 % y 87,4 %. El modelo del sensor no arregla unas poses malas: las paredes dobles las quita el cierre de bucle, no los log-odds.

---

## Para llevarse de esta sesión

- **El SLAM es circular**: para hacer el mapa hace falta la pose, y para saber la pose hace falta el mapa. Lo rompe volver a ver algo conocido.
- **EKF-SLAM** mete las balizas en el estado (3 + 2N): todo se corrige junto, pero Σ crece con el cuadrado de N.
- **La asociación de datos** es lo más delicado: una sola medida mal atribuida rompe el mapa para siempre; la puerta de Mahalanobis la descarta, pero deja fuera el 1 % de las buenas, que crean copias.
- **El grafo de poses** es SLAM completo por mínimos cuadrados, con el LM de S10: el cierre de bucle reparte la deriva por toda la vuelta, y un cierre falso lo rompe todo; lo delata la suma de los residuos, no el suyo.
- **La rejilla de ocupación** es una puerta de Bayes por celda, en log-odds; con las poses del grafo, es el mapa del que parte S22.

*Cuaderno del curso 82514 Mecatrónica y Robótica · IQS Universitat Ramon Llull · curso 2026/27*

*© Guillermo Reyes Carmenaty · Publicado bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es): puedes usarlo, adaptarlo y redistribuirlo, incluso con fines comerciales, siempre que reconozcas la autoría.*